# Y4 + Y5 · Arabic twins and Arabic experiments (Youssef) · GPU runtime
**Runtime > Change runtime type > T4 GPU** (or L4/A100 if you have it), then **Runtime > Run all** (~1.5-2 h on a T4). Keep the tab open.

1. Y4: Arabic twins `_A _M _B_A _C_A` for 200 test sources (Qwen3-8B translation + name glossary)
2. Y5: zero-shot NLI + reranker scores for them, the H5 verdict heads and Feba's baselines on Arabic twins + X-FACT AR test → `results/Y5/arabic_E7.csv`
3. Downloads `results_Y4_Y5.zip` at the end.

In [ ]:
!git clone -q https://github.com/himaallu/Group-Project.git /content/Group-Project
%cd /content/Group-Project
!test -f src/eval/run_y5_ar.py && echo OK code found || echo 'STOP: merge the Y4-Y5 PR first'
!pip install -q -U transformers accelerate bitsandbytes lightgbm sentencepiece

## Y4 · quick test on 5 sources (check the Arabic looks right)

In [ ]:
!python -m src.copies.arabic_twins --n 5
import json
for l in open('data/copies/ar/test.jsonl').readlines()[:4]:
    r = json.loads(l); print(r['id'], r['label'], r['language'], '|', r['claim'], '|', r['evidence'][0]['text'][:150])

## Y4 · all 200 sources (~1.5 h on a T4; the 5 above come from the cache)

In [ ]:
!python -m src.copies.arabic_twins

## Y5 · NLI and relevance scores for the Arabic twins (~5 min)

In [ ]:
!python -m src.nli.score_zeroshot data/copies/ar/test.jsonl --out-dir features/nli_zeroshot/copies_ar
!python -m src.retrieval.rerank data/copies/ar/test.jsonl --model RathiAbeth/twincheck-reranker-v1 --out features/relevance_copies_ar.parquet

## Y5 · baselines on Arabic twins + X-FACT AR test (CPU, ~5 min)

In [ ]:
!mkdir -p results/Y5/baselines
!python -m src.eval.baselines.zeroshot_nli --val-records data/processed/en/multifc_val.jsonl --val-nli features/nli_zeroshot/multifc_val.parquet --predict data/copies/ar/test.jsonl data/processed/ar/xfact_ar_test.jsonl --predict-nli features/nli_zeroshot/copies_ar/test.parquet features/nli_zeroshot/xfact_ar_test.parquet --out results/Y5/baselines/preds_zeroshot_nli.jsonl
!python -m src.eval.baselines.similarity --train data/processed/en/multifc_train.jsonl --val data/processed/en/multifc_val.jsonl --predict data/copies/ar/test.jsonl data/processed/ar/xfact_ar_test.jsonl --out-dir results/Y5/baselines --no-wandb

## Y5 · verdict heads + table

In [ ]:
!python -m src.eval.run_y5_ar

In [ ]:
!zip -q -r /content/results_Y4_Y5.zip data/copies/ar data/translate/glossary_en_ar.csv results/Y4 results/Y5 features/nli_zeroshot/copies_ar features/relevance_copies_ar.parquet
from google.colab import files; files.download('/content/results_Y4_Y5.zip')